In [36]:

#!pip install langchain langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb
"""langchain langchain-core langchain-community are the components of the langchain 
where as pypdf is used to extract data from pdf and pymupdf is used to extract data
from a complex pdf which has imges and other things. sentence transformers are used to create embeddings
there are many more transformers but we have used this and the vector database used will be the chroma database""" 

'langchain langchain-core langchain-community are the components of the langchain \nwhere as pypdf is used to extract data from pdf and pymupdf is used to extract data\nfrom a complex pdf which has imges and other things. sentence transformers are used to create embeddings\nthere are many more transformers but we have used this and the vector database used will be the chroma database'

In [37]:
#!python.exe -m pip install --upgrade pip

In [38]:
from langchain_core.documents import Document

In [39]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://www.google.com"}
)

In [40]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [41]:
# Text Data
from langchain_community.document_loaders.text import TextLoader
#alag alag data ke liye we have different loaders for text,pdf etc
loader = TextLoader("Python.txt", encoding="utf-8")

In [42]:
document = loader.load()

In [43]:
document

[Document(metadata={'source': 'Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and more.\n

In [44]:
# PDF data
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("research.pdf")

document = pdf_loader.load()
# we can perform the same operation by PyMuPDFLoader as well
# document

### Ingestion Pipeline

In [45]:
# Data => Documents 
import os # so that we can access the file whereever in the device
from langchain_community.document_loaders.pdf import PyPDFLoader

In [46]:
def load_all_pdfs():
    folder_path = "pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [47]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [48]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

### chunks

In [49]:
# chunks
#!pip install langchain_text_splitters

In [50]:
# chunk size is max number of characters chunk can have
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

    

In [51]:
chunks = split_docs(all_pdf_documents)

In [52]:
len(chunks)

320

### Embeddings

In [53]:
# sentence transformers are python framework in which we can use multiple embedding models
from sentence_transformers import SentenceTransformer

In [54]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model...", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("embeddings shape", embeddings.shape)
        return embeddings

In [55]:
embedding_manager = EmbeddingManager()

loading model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\anuj1234\AppData\Local\Temp\ipykernel_19948\946935876.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Vector Store

In [84]:
import chromadb
import uuid #it is similar to as giving the index to the data in sql here it gives index to the documents

In [85]:
# persist directory matlab jitne bhi vectors hi yah jitne bhi documents hi woh kaha store honge
# collection store ka matlab hai kya woh table ka naam hoga yah woh files ka naam hoga 
# vector store is noting but a smaller cersion or type of vector database as we are creating it for our personal project
# companies mein we use vector database
class VectorStoreManager:
    def __init__(self, persist_directory="vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None # collection ko abhi tak create nahi kiya hai isliye collection ko none naam dena padega 
        self.client = None  # jaise sql mein humein client chahiye hota hai user se interact kareke liye so that data can be updated and stored in the database

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collections:", self.collection_name)
        print("docs in collection:", self.collection.count())
        
        
    def add_documents(self, documents, embeddings):  # vector store mein we store embeddings + chunks(documents)
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match number of embeddings")

        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        """embedding and document are two different data struture par jab bhi humein multiple datastructure se common 
        info extract karni ho then we zip them and we collectivelty run a loop on them"""

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # doc is an individual document in individual chunk 
            # each document has its own page content and metadata
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)  # converted in the form of dictonary
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)


            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            # collection mein charo value ko ek singe unit ke tarah apne collection ke andar insert karna hai 

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )


        print("total documents added in vector store", len(documents_content))
        print("docs in collection:", self.collection.count())
            
            

In [86]:
vector_store = VectorStoreManager()

initialized the vector store with collections: pdf_documents
docs in collection: 1920


In [87]:
# every chunk is a document in itself, a chunk has 320 documents
# data => documents => chunks => embeddings => store in vector store 

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape (320, 384)
total documents added in vector store 320
docs in collection: 2240


### Retrieval Pipeline

In [88]:
from sklearn.metrics.pairwise import cosine_similarity

In [89]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]    # we have it in the form of list we have mentioned it in [] brackets

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k    
        )

        # cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]


            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance


                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "document" : document,
                        "distance" : distance,
                        "metadata" : metadata,
                        "similarity_score" : similarity_score,
                        "rank" : i+1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs
            
            
        

In [90]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [91]:
rag_retriever.retrieve("What is encoder decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 5 documents


[{'id': 'doc_41583bbc-7687-4cc0-8e18-a165941d601d',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'distance': 0.7716919779777527,
  'metadata': {'published': '2017',
   'publisher': 'Curran Associates, Inc.',
   'date': '2017',
   'language': 'en-US',
   'type': 'Conference Proceedings',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'title': 'Attention is All you Need',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',


### Integrate with LLMs

# Groq

In [120]:
API_Key_GROQ = "Enter your api key"


In [121]:
#!pip install langchain-groq

In [122]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model = "qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
    
)

In [125]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context = query
    prompt = f"""use given context to generate the answer for the query
                Context: {context}
                Query: {query}
                """
    response = llm.invoke([prompt.format(context=context, query=query)])  
    return response.content

In [126]:
answer = generate_output("What is encoder-decoder?", rag_retriever, llm)  

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape (1, 384)
retrieved 3 documents


In [127]:
print(answer)

<think>
Okay, the user is asking, "What is encoder-decoder?" Let me start by recalling what I know about encoder-decoder architectures. From the context provided, it's talking about layers in both the encoder and decoder, specifically mentioning that the decoder has three sub-layers, including a multi-head attention over the encoder's output. The encoder has two sub-layers, and both use residual connections and layer normalization.

First, I need to define what an encoder-decoder architecture is in general. It's a framework used in neural networks, especially in sequence-to-sequence tasks like machine translation. The encoder processes the input sequence and encodes it into a context vector, while the decoder generates the output sequence based on that context.

Looking at the context, the encoder is a stack of N=6 identical layers. Each encoder layer has two sub-layers: maybe self-attention and a feed-forward network? The decoder also has N=6 layers but adds a third sub-layer that att

In [128]:
# Claude
!pip install langchain-anthropic

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/929.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/929.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/929.8 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/929.8 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 929.8/929.8 kB 1.3 MB/s  0:00:00

   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------